# ⏰ Notebook 05 — Delta Time Travel & Audit Trail

**Goal:** Query historical snapshots of Delta Tables, audit changes, and demonstrate regulatory audit trail capabilities.

> **Run time:** ~5 min

## What is Time Travel?
Delta Lake automatically records every write operation in a transaction log. You can query any previous version of the data — even after it's been updated or deleted.

```sql
-- Query the table as it was 2 hours ago
SELECT * FROM fact_loans TIMESTAMP AS OF '2025-01-01 10:00:00'

-- Query version 3 specifically
SELECT * FROM fact_loans VERSION AS OF 3
```

## Banking Regulatory Use Cases
| Requirement | Delta Time Travel Solution |
|-------------|---------------------------|
| OCC exam: "Show me loan data as of Q3 close" | `VERSION AS OF` or `TIMESTAMP AS OF` |
| Audit: "Who changed LOAN0001 and when?" | `DESCRIBE HISTORY` |
| Incident: "Accidentally deleted records — restore" | `RESTORE TABLE` |
| Compliance: "Prove data hasn't been altered" | Transaction log is append-only |
| DFAST: "Reproduce stress test inputs" | Exact snapshot of data used |

In [ ]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F

# View full history of fact_loans
print('fact_loans transaction history:')
DeltaTable.forName(spark, 'fact_loans').history() \
    .select('version','timestamp','operation','operationParameters','operationMetrics') \
    .show(truncate=False)

## Step 1 — Query a Specific Historical Version

In [ ]:
%%sql
-- Version 0 = the original load from Demo 05
-- Version 1 = after the MERGE corrections in Notebook 02
SELECT LoanID, LoanStatus, OutstandingBalance
FROM fact_loans VERSION AS OF 0
WHERE LoanID IN ('LOAN0001','LOAN0050','LOAN0501','LOAN0502')
ORDER BY LoanID

In [ ]:
%%sql
-- Current version (after MERGE)
SELECT LoanID, LoanStatus, OutstandingBalance
FROM fact_loans
WHERE LoanID IN ('LOAN0001','LOAN0050','LOAN0501','LOAN0502')
ORDER BY LoanID

## Step 2 — Audit: What Changed Between Versions?

In [ ]:
# Show what changed in LOAN0001 between version 0 and current
v0 = spark.read.format('delta').option('versionAsOf', 0).table('fact_loans') \
    .filter('LoanID = "LOAN0001"') \
    .select('LoanID', F.col('LoanStatus').alias('Status_v0'), F.col('OutstandingBalance').alias('Balance_v0'))

v_current = spark.table('fact_loans') \
    .filter('LoanID = "LOAN0001"') \
    .select('LoanID', F.col('LoanStatus').alias('Status_current'), F.col('OutstandingBalance').alias('Balance_current'))

audit = v0.join(v_current, on='LoanID')
audit.show()

row = audit.collect()[0]
if row['Status_v0'] != row['Status_current']:
    print(f'⚠️  LOAN0001 status changed: {row["Status_v0"]} → {row["Status_current"]}')
if row['Balance_v0'] != row['Balance_current']:
    print(f'💰 LOAN0001 balance changed: ${row["Balance_v0"]:,.2f} → ${row["Balance_current"]:,.2f}')

## Step 3 — TIMESTAMP AS OF: Quarter-End Snapshot

In [ ]:
# Get the timestamp of version 0 (original load)
history = DeltaTable.forName(spark, 'fact_loans').history(10).collect()
v0_ts = [r for r in history if r['version'] == 0][0]['timestamp']
v0_ts_str = v0_ts.strftime('%Y-%m-%d %H:%M:%S')

print(f'Original load timestamp: {v0_ts_str}')

# Query using timestamp (regulatory use: reproduce Q-end data)
df_snapshot = spark.read.format('delta') \
    .option('timestampAsOf', v0_ts_str) \
    .table('fact_loans')

print(f'\nSnapshot row count at {v0_ts_str}: {df_snapshot.count()}')
df_snapshot.groupBy('LoanStatus').count().orderBy('count', ascending=False).show()

## Step 4 — RESTORE TABLE: Undo an Accidental Change

In [ ]:
# Simulate an accidental bulk update
print('Simulating accidental bulk update...')
spark.sql("UPDATE fact_loans SET LoanStatus = 'CORRUPTED' WHERE LoanStatus = 'Current'")
corrupted_count = spark.sql("SELECT COUNT(*) AS n FROM fact_loans WHERE LoanStatus = 'CORRUPTED'").collect()[0]['n']
print(f'Accidentally corrupted {corrupted_count} records!')

In [ ]:
# Get the last good version number (before the accidental update)
history = DeltaTable.forName(spark, 'fact_loans').history(3).collect()
last_good_version = [r for r in history if r['operation'] != 'UPDATE'][0]['version']
print(f'Restoring to version {last_good_version}...')

spark.sql(f'RESTORE TABLE fact_loans TO VERSION AS OF {last_good_version}')

restored = spark.sql("SELECT COUNT(*) AS n FROM fact_loans WHERE LoanStatus = 'CORRUPTED'").collect()[0]['n']
print(f'\n✅ Restore complete. Corrupted records remaining: {restored}')
spark.sql("SELECT LoanStatus, COUNT(*) AS Count FROM fact_loans GROUP BY LoanStatus ORDER BY Count DESC").show()

## Step 5 — Retention Policy

In [ ]:
%%sql
-- Set retention to 90 days for regulatory compliance (default is 7 days)
ALTER TABLE fact_loans
SET TBLPROPERTIES ('delta.logRetentionDuration' = 'interval 90 days',
                   'delta.deletedFileRetentionDuration' = 'interval 90 days')

In [ ]:
%%sql
DESCRIBE DETAIL fact_loans